In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from transformers import AutoTokenizer, VitsModel

# MMS-TTS Bulgarian model.
MMS_MODEL_ID = "facebook/mms-tts-bul"
MMS_CACHE_DIR = Path("/home/anna/python/MIPT/speach_recognition/FP/models/mms_tts")
MMS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Try local cache first, then allow download.
try:
    mms_tokenizer = AutoTokenizer.from_pretrained(
        MMS_MODEL_ID,
        cache_dir=str(MMS_CACHE_DIR),
        local_files_only=True,
    )
    mms_model = VitsModel.from_pretrained(
        MMS_MODEL_ID,
        cache_dir=str(MMS_CACHE_DIR),
        local_files_only=True,
    )
    loaded_mode = "local cache"
except Exception:
    mms_tokenizer = AutoTokenizer.from_pretrained(
        MMS_MODEL_ID,
        cache_dir=str(MMS_CACHE_DIR),
    )
    mms_model = VitsModel.from_pretrained(
        MMS_MODEL_ID,
        cache_dir=str(MMS_CACHE_DIR),
    )
    loaded_mode = "downloaded"

mms_device = "cuda" if torch.cuda.is_available() else "cpu"
mms_model = mms_model.to(mms_device)
mms_model.eval()

MMS_NATIVE_SR = int(getattr(mms_model.config, "sampling_rate", 16000))

print(f"MMS model loaded ({loaded_mode})")
print(f"Model id: {MMS_MODEL_ID}")
print(f"Device: {mms_device}")
print(f"Native sample rate: {MMS_NATIVE_SR}")

config.json:   0%|          | 0.00/1.64k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

MMS model loaded (downloaded)
Model id: facebook/mms-tts-bul
Device: cuda
Native sample rate: 16000


In [13]:
from pathlib import Path
import re
import numpy as np
import soundfile as sf
import librosa
import torch

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else None

# === Paths for MMS output ===
TEXT_PATH = Path("/home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data/prepared/combined_bg_for_asr.txt")
AUDIO2_DIR = Path("/home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data/audio2")
TARGET_SR = 16000

# Reuse same filtering strategy.
MIN_LINE_CHARS = 10
MAX_LINE_CHARS = 260


def load_and_filter_lines(path: Path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = []
    for raw in text.splitlines():
        line = re.sub(r"\s+", " ", raw).strip()
        if not line:
            continue
        if len(line) < MIN_LINE_CHARS or len(line) > MAX_LINE_CHARS:
            continue
        lines.append(line)
    return lines


if "mms_model" not in globals() or "mms_tokenizer" not in globals():
    raise RuntimeError("Run previous MMS model-load cell first.")

if not TEXT_PATH.exists():
    raise FileNotFoundError(f"Prepared text file not found: {TEXT_PATH}")

AUDIO2_DIR.mkdir(parents=True, exist_ok=True)

lines = load_and_filter_lines(TEXT_PATH)
if not lines:
    raise RuntimeError("No valid lines for MMS synthesis after filtering.")

metadata_path = AUDIO2_DIR / "metadata_mms.csv"

print(f"MMS synthesis started for {len(lines)} lines...")
written = 0

with metadata_path.open("w", encoding="utf-8") as meta_f:
    iterator = tqdm(lines, total=len(lines), desc="MMS synth", unit="utt", dynamic_ncols=True)
    for idx, text_line in enumerate(iterator, 1):
        inputs = mms_tokenizer(text_line, return_tensors="pt")
        inputs = {k: v.to(mms_device) for k, v in inputs.items()}

        with torch.no_grad():
            waveform = mms_model(**inputs).waveform.squeeze(0).detach().cpu().numpy()

        waveform = np.asarray(waveform, dtype=np.float32)

        # Convert sample rate if model native SR differs.
        if MMS_NATIVE_SR != TARGET_SR:
            waveform = librosa.resample(waveform, orig_sr=MMS_NATIVE_SR, target_sr=TARGET_SR)

        out_name = f"mms_bg_{idx:06d}.wav"
        out_path = AUDIO2_DIR / out_name
        sf.write(out_path, waveform, TARGET_SR, subtype="PCM_16")

        meta_f.write(f"{out_name}|{text_line}\n")
        written += 1

print("Done.")
print(f"Audio dir: {AUDIO2_DIR}")
print(f"Generated wav files: {written}")
print(f"Metadata: {metadata_path}")

generated_audio2_count = written

MMS synthesis started for 12858 lines...


MMS synth:   0%|          | 0/12858 [00:00<?, ?utt/s]

Done.
Audio dir: /home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data/audio2
Generated wav files: 12858
Metadata: /home/anna/python/MIPT/speach_recognition/FP/data/sintetyc_data/audio2/metadata_mms.csv


In [1]:
!tree /home/anna/python/MIPT/speach_recognition/FP/data/alice

/home/anna/python/MIPT/speach_recognition/FP/data/alice
├── 0001.wav
├── 0002.wav
├── 0003.wav
├── 0004.wav
├── 0005.wav
├── 0006.wav
├── 0007.wav
├── 0008.wav
├── 0009.wav
├── 0010.wav
├── 0011.wav
├── 0012.wav
├── 0013.wav
├── 0014.wav
├── 0015.wav
├── 0016.wav
├── 0017.wav
├── 0018.wav
├── 0019.wav
├── 0020.wav
├── 0021.wav
├── 0022.wav
├── 0023.wav
├── 0024.wav
├── 0025.wav
├── 0026.wav
├── 0027.wav
├── 0028.wav
├── 0029.wav
├── 0030.wav
├── 0031.wav
├── 0032.wav
├── 0033.wav
├── 0034.wav
├── 0035.wav
├── 0036.wav
├── 0037.wav
├── 0038.wav
├── 0039.wav
├── 0040.wav
├── 0041.wav
├── 0042.wav
├── 0043.wav
├── 0044.wav
├── 0045.wav
├── 0046.wav
├── 0047.wav
├── 0048.wav
├── 0049.wav
├── 0050.wav
├── 0051.wav
├── 0052.wav
├── 0053.wav
├── 0054.wav
├── 0055.wav
├── 0056.wav
├── 0057.wav
├── 0058.wav
├── 0059.wav
├── 0060.wav
├── 0061.wav
├── 0062.wav
├── 0063.wav
├── 0064.wav
├── 0065.wav
├── 0066.wav
├── 0067.wav
├── 0068.wav
├── 0069.wav
├── 0070.wav
├── 0071.wav
├── 0072.wav
├── 0073